# Ray Unit 3 - Sharded Data

Pandas puts an entire DF in RAM. What if the data is too big?

Ray shards the data into blocks while keeping a pandas-like API.

Popular alternatives: Spark, Dask, Daft.

## Setup

Use the same `22971-ray` environment and notebook kernel as the previous units.

In [1]:
import math
import time
import numpy as np
import pandas as pd
import ray

from pathlib import Path
from IPython.display import display
from ray.data.aggregate import AggregateFnV2, Count, Mean, Sum
from ray.data.block import BlockAccessor

ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False, log_to_driver=False, logging_level='ERROR')
print('Ray version:', ray.__version__)
print('Ray resources:', ray.cluster_resources())

/Users/shlomibenshushan/miniconda3/envs/22971-ray/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-20 09:01:09,191	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-04-20 09:01:09,385	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Ray version: 2.54.1
Ray resources: {'node:__internal_head__': 1.0, 'memory': 8351350784.0, 'node:127.0.0.1': 1.0, 'CPU': 12.0, 'object_store_memory': 2147483648.0}


/Users/shlomibenshushan/miniconda3/envs/22971-ray/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## 0. Generate Data

We generate synthetic gameplay telemetry data with these columns:

    | Column | Type |
    |---|---|
    | `event_ts` | `timestamp[ns]` |
    | `player_id` | `int32` |
    | `session_id` | `int32` |
    | `region` | `string` |
    | `platform` | `string` |
    | `event_type` | `string` |
    | `level_id` | `int16` |
    | `latency_ms` | `int32` |
    | `spend_usd` | `float` |
    | `is_test_user` | `bool` |
    | `payload_blob` | `string` |

- `payload_blob` is a long nonsense string that makes each row artificially wide.
- We save the dataset to `...\fake_data\part-XX.parquet`.
- The dataset is split into `NUM_SHARDS` parquet shards.

### Helper functions

Constants for synthetic data generation

In [2]:
SEED = 0
N_ROWS = 60_000
NUM_SHARDS = 8
PAYLOAD_CHARS = 2_000  # 2 KB per row
DATA_DIR = 'fake_data'
OUT_DIR = 'output'

Categorical options for synthetic data

In [3]:
REGIONS = np.array(['NA', 'EU', 'APAC', 'LATAM', 'MEA'])
PLATFORMS = np.array(['pc', 'console', 'mobile'])
EVENT_TYPES = np.array(['match_start', 'match_end', 'quest', 'purchase', 'crash', 'chat'])

For reproducibility, we generate the same payload blobs for each row, regardless of the random seed

In [4]:
def build_payload_column(n_rows: int, payload_chars: int = PAYLOAD_CHARS) -> pd.Series:
    seeds = pd.Series(np.arange(n_rows), dtype='int64').map(
        lambda i: f'blob-{i:07d}-{(i * 2654435761) & 0xFFFFFFFF:08x}'
    )
    return seeds.str.pad(payload_chars, side='right', fillchar='x')

In [5]:
# Example:
payload_column = build_payload_column(5, 20)
payload_column

0    blob-0000000-00000000
1    blob-0000001-9e3779b1
2    blob-0000002-3c6ef362
3    blob-0000003-daa66d13
4    blob-0000004-78dde6c4
dtype: object

Build a synthetic telemetry DataFrame with realistic distributions and correlations.

In [6]:
def build_telemetry_frame(n_rows: int = N_ROWS, seed: int = SEED, payload_chars: int = PAYLOAD_CHARS) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    event_type = rng.choice(EVENT_TYPES, size=n_rows, p=[0.28, 0.22, 0.25, 0.07, 0.03, 0.15])
    event_ts = pd.Timestamp('2025-01-01') + pd.to_timedelta(rng.integers(0, 14 * 24 * 60 * 60, size=n_rows), unit='s')
    latency_ms = np.round(rng.gamma(shape=2.4, scale=32.0, size=n_rows) + rng.integers(5, 25, size=n_rows)).astype('int32')
    latency_ms[event_type == 'crash'] *= 2

    spend_usd = np.zeros(n_rows, dtype='float32')
    purchase_mask = event_type == 'purchase'
    spend_usd[purchase_mask] = rng.uniform(0.99, 59.99, size=purchase_mask.sum()).round(2)

    frame = pd.DataFrame({
        'event_ts': event_ts,
        'player_id': rng.integers(10_000, 90_000, size=n_rows, dtype=np.int32),
        'session_id': rng.integers(100_000, 900_000, size=n_rows, dtype=np.int32),
        'region': rng.choice(REGIONS, size=n_rows, p=[0.36, 0.27, 0.22, 0.10, 0.05]),
        'platform': rng.choice(PLATFORMS, size=n_rows, p=[0.42, 0.18, 0.40]),
        'event_type': event_type,
        'level_id': rng.integers(1, 60, size=n_rows, dtype=np.int16),
        'latency_ms': latency_ms,
        'spend_usd': spend_usd,
        'is_test_user': rng.random(n_rows) < 0.02,
        'payload_blob': build_payload_column(n_rows, payload_chars),
    })

    return frame.reset_index(drop=True)

In [7]:
# Example:
frame = build_telemetry_frame(5, payload_chars=20)
frame

,event_ts,player_id,session_id,region,platform,event_type,level_id,latency_ms,spend_usd,is_test_user,payload_blob
0,2025-01-10 02:12:13,10428,617751,MEA,pc,quest,4,58,0.00,False,blob-0000000-00000000
1,2025-01-13 18:41:09,19942,305839,MEA,pc,match_start,25,47,0.00,False,blob-0000001-9e3779b1
2,2025-01-08 01:13:07,10662,592308,APAC,mobile,match_start,42,16,0.00,False,blob-0000002-3c6ef362
3,2025-01-09 11:49:46,63649,711243,APAC,console,match_start,29,38,0.00,False,blob-0000003-daa66d13
4,2025-01-14 14:10:10,52049,406942,APAC,pc,purchase,42,57,2.66,False,blob-0000004-78dde6c4


For demonstration purposes, we build a small dataset and write it out as Parquet shards

In [8]:
def write_parquet_shards(frame: pd.DataFrame, out_dir: str, num_shards: int = NUM_SHARDS) -> None:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows_per_shard = math.ceil(len(frame) / num_shards)
    for shard_id in range(num_shards):
        start = shard_id * rows_per_shard
        stop = min((shard_id + 1) * rows_per_shard, len(frame))
        shard = frame.iloc[start:stop]
        if not shard.empty:
            shard.to_parquet(out_dir / f'part-{shard_id:02d}.parquet', index=False)

Enrichment function to add derived columns to the telemetry data

In [9]:
def enrich_batch(batch: pd.DataFrame) -> pd.DataFrame:
    batch = batch.copy()
    batch['hour_of_day'] = batch['event_ts'].dt.hour.astype('int8')
    batch['is_purchase'] = batch['event_type'].eq('purchase')
    batch['is_crash'] = batch['event_type'].eq('crash')
    batch['latency_band'] = pd.cut(
        batch['latency_ms'],
        bins=[-1, 60, 120, 240, np.inf],
        labels=['fast', 'stable', 'slow', 'spiky'],
    ).astype('string')
    return batch

In [10]:
# Example:
enriched_frame = enrich_batch(frame)
enriched_frame

,event_ts,player_id,session_id,region,platform,event_type,level_id,latency_ms,spend_usd,is_test_user,payload_blob,hour_of_day,is_purchase,is_crash,latency_band
0,2025-01-10 02:12:13,10428,617751,MEA,pc,quest,4,58,0.00,False,blob-0000000-00000000,2,False,False,fast
1,2025-01-13 18:41:09,19942,305839,MEA,pc,match_start,25,47,0.00,False,blob-0000001-9e3779b1,18,False,False,fast
2,2025-01-08 01:13:07,10662,592308,APAC,mobile,match_start,42,16,0.00,False,blob-0000002-3c6ef362,1,False,False,fast
3,2025-01-09 11:49:46,63649,711243,APAC,console,match_start,29,38,0.00,False,blob-0000003-daa66d13,11,False,False,fast
4,2025-01-14 14:10:10,52049,406942,APAC,pc,purchase,42,57,2.66,False,blob-0000004-78dde6c4,14,True,False,fast


### Data generation

Check for existing parquet shards to avoid regenerating data on every run

In [11]:
workspace_dir = Path.cwd()
input_dir = workspace_dir / DATA_DIR
output_dir = workspace_dir / OUT_DIR

workspace_dir.mkdir(parents=True, exist_ok=True)

existing_shards = sorted(input_dir.glob('part-*.parquet'))
if existing_shards:
    print(f"Found {len(existing_shards)} existing parquet shards in {input_dir}; skipping generation.")
    telemetry_pdf = pd.concat((pd.read_parquet(path) for path in existing_shards), ignore_index=True)
else:
    telemetry_pdf = build_telemetry_frame()
    write_parquet_shards(telemetry_pdf, input_dir, num_shards=NUM_SHARDS)

display(telemetry_pdf.head())
size_bytes = telemetry_pdf.memory_usage(deep=True).sum()
print(f"size: {size_bytes / 1024**2:.2f} MiB")

Found 8 existing parquet shards in /Users/shlomibenshushan/Repositories/OU-22971-Toolbox/Ray/3_ray_data/fake_data; skipping generation.


,event_ts,player_id,session_id,region,platform,event_type,level_id,latency_ms,spend_usd,is_test_user,payload_blob
0,2025-01-05 04:02:26,81408,502575,APAC,pc,quest,49,53,0.00,False,blob-0000000-00000000xxxxxxxxxxxxxxxxxxxxxxxxx...
1,2025-01-10 17:00:11,60826,556316,APAC,console,match_start,44,129,0.00,False,blob-0000001-9e3779b1xxxxxxxxxxxxxxxxxxxxxxxxx...
2,2025-01-07 21:20:43,82367,334510,EU,pc,match_start,20,100,0.00,False,blob-0000002-3c6ef362xxxxxxxxxxxxxxxxxxxxxxxxx...
3,2025-01-03 19:40:54,41559,168491,EU,pc,match_start,1,143,0.00,False,blob-0000003-daa66d13xxxxxxxxxxxxxxxxxxxxxxxxx...
4,2025-01-14 06:36:17,59378,497715,EU,mobile,purchase,7,133,2.09,False,blob-0000004-78dde6c4xxxxxxxxxxxxxxxxxxxxxxxxx...


size: 128.05 MiB


## 1. Ray Data Mental Model

#### A Ray `Dataset` feels a lot like a pandas `DataFrame` for the operations in this notebook.

Load the data into a Ray Dataset, specifying the number of output blocks to control parallelism

In [12]:
events_ds = ray.data.read_parquet(str(input_dir), override_num_blocks=NUM_SHARDS)
events_ds.show(5)

Parquet dataset sampling: 100%|██████████| 2.00/2.00 [00:00<00:00, 2.98 file/s]
2026-04-20 09:01:13,832	INFO parquet_datasource.py:1079 -- Estimated parquet encoding ratio is 14.810.
2026-04-20 09:01:13,833	INFO parquet_datasource.py:1139 -- Estimated parquet reader batch size at 65218 rows
2026-04-20 09:01:14,523	INFO dataset.py:3670 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-04-20 09:01:14,544	INFO logging.py:392 -- Registered dataset logger for dataset dataset_1_0
2026-04-20 09:01:14,560	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_1_0. Full logs are in /tmp/ray/session_2026-04-20_09-01-09_472804_15805/logs/ray-data
2026-04-20 09:01:14,561	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=5]
2026-04-20 09:01:14,565	WARNING resource_manager.py:141 -- ⚠️  Ray's object store is configured

{'event_ts': Timestamp('2025-01-05 04:02:26'), 'player_id': 81408, 'session_id': 502575, 'region': 'APAC', 'platform': 'pc', 'event_type': 'quest', 'level_id': 49, 'latency_ms': 53, 'spend_usd': 0.0, 'is_test_user': False, 'payload_blob': 'blob-0000000-00000000xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx

#### The dataset is sharded into **blocks**: each block is stored as a separate object in Ray's object store.

In [13]:
# We don't need to know what ref_bundles are (ray.data internals)
for bundle_idx, ref_bundle in enumerate(events_ds.iter_internal_ref_bundles()):
    [(block_ref, metadata)] = ref_bundle.blocks
    print(f"block {bundle_idx}: ref={block_ref}, rows={metadata.num_rows}, size={metadata.size_bytes / 1024**2:.2f} MiB")

2026-04-20 09:01:15,661	INFO logging.py:392 -- Registered dataset logger for dataset dataset_0_0
2026-04-20 09:01:15,664	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_0_0. Full logs are in /tmp/ray/session_2026-04-20_09-01-09_472804_15805/logs/ray-data
2026-04-20 09:01:15,665	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_0_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet]
2026-04-20 09:01:15,685	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_0_0 =======
2026-04-20 09:01:15,686	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-04-20 09:01:15,686	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/1.0GiB object store
2026-04-20 09:01:15,687	INFO logging_progress.py:181 -- 
2026-04-20 09:01:15,688	INFO logging_progress.py:231 -- ReadParquet: 0/1
2026-04-20 09:01:15,688	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 7 (0.0B); Resources: 1.0 CPU, 384.0MiB obje

block 0: ref=ObjectRef(4bd8bcae5a37a5edffffffffffffffffffffffff0100000002000000), rows=7500, size=14.72 MiB
block 1: ref=ObjectRef(a46e4f16dd9f5b94ffffffffffffffffffffffff0100000002000000), rows=7500, size=14.72 MiB
block 2: ref=ObjectRef(7112efd42def1f0cffffffffffffffffffffffff0100000002000000), rows=7500, size=14.72 MiB
block 3: ref=ObjectRef(38e9ebe6c83cf76effffffffffffffffffffffff0100000002000000), rows=7500, size=14.72 MiB


2026-04-20 09:01:16,602	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_0_0 execution finished in 0.94 seconds


block 4: ref=ObjectRef(df62676e7891ea4affffffffffffffffffffffff0100000002000000), rows=7500, size=14.72 MiB
block 5: ref=ObjectRef(a50fa2c37a366af6ffffffffffffffffffffffff0100000002000000), rows=7500, size=14.72 MiB
block 6: ref=ObjectRef(9612d4d53579c9f0ffffffffffffffffffffffff0100000002000000), rows=7500, size=14.72 MiB
block 7: ref=ObjectRef(8a87702593e11939ffffffffffffffffffffffff0100000002000000), rows=7500, size=14.72 MiB


#### Execution is **lazy**: no data is manipulated unless explicitly required.

In [14]:
sorted_ds = events_ds.sort(key='event_ts')

Why did this cell run instantly?

Ray Data built a plan:
1. Read the data from the source files, with different tasks reading different blocks.
2. Sort the rows by `event_ts`.

**It did not execute that plan yet.**


In [15]:
sorted_ds.explain()


-------- Logical Plan --------
Sort[Sort]
+- Read[ReadParquet]

-------- Logical Plan (Optimized) --------
Sort[Sort]
+- Read[ReadParquet]

-------- Physical Plan --------
AllToAllOperator[Sort]
+- TaskPoolMapOperator[ReadParquet]
   +- InputDataBuffer[Input]

-------- Physical Plan (Optimized) --------
AllToAllOperator[Sort]
+- TaskPoolMapOperator[ReadParquet]
   +- InputDataBuffer[Input]



We can tell Ray to execute the plan explicitly.

In [16]:
events_materialized = sorted_ds.materialize()
events_materialized

2026-04-20 09:01:16,637	INFO logging.py:392 -- Registered dataset logger for dataset dataset_3_0
2026-04-20 09:01:16,640	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_3_0. Full logs are in /tmp/ray/session_2026-04-20_09-01-09_472804_15805/logs/ray-data
2026-04-20 09:01:16,640	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_3_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> AllToAllOperator[Sort]
2026-04-20 09:01:16,657	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_3_0 =======
2026-04-20 09:01:16,658	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-04-20 09:01:16,659	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/1.0GiB object store
2026-04-20 09:01:16,659	INFO logging_progress.py:181 -- 
2026-04-20 09:01:16,660	INFO logging_progress.py:231 -- ReadParquet: 0/1
2026-04-20 09:01:16,660	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 7 (0.0B); Resourc

shape: (60000, 11)
╭─────────────────────┬───────────┬────────────┬────────┬──────────┬─────────────┬──────────┬────────────┬───────────┬──────────────┬──────────────────────────────────────────╮
│ event_ts            ┆ player_id ┆ session_id ┆ region ┆ platform ┆ event_type  ┆ level_id ┆ latency_ms ┆ spend_usd ┆ is_test_user ┆ payload_blob                             │
│ ---                 ┆ ---       ┆ ---        ┆ ---    ┆ ---      ┆ ---         ┆ ---      ┆ ---        ┆ ---       ┆ ---          ┆ ---                                      │
│ timestamp[ns]       ┆ int32     ┆ int32      ┆ string ┆ string   ┆ string      ┆ int16    ┆ int32      ┆ float     ┆ bool         ┆ string                                   │
╞═════════════════════╪═══════════╪════════════╪════════╪══════════╪═════════════╪══════════╪════════════╪═══════════╪══════════════╪══════════════════════════════════════════╡
│ 2025-01-01 00:00:04 ┆ 50679     ┆ 723835     ┆ LATAM  ┆ console  ┆ chat        ┆ 22       ┆ 24

- Some methods trigger execution (e.g. `show()`, `sum()`, `materialize()`).
- Some methods are lazy (e.g. `sort()`, `groupby()`, `groupby(...).aggregate()`).

If a method returns another `Dataset` or `GroupedData`, it is usually still lazy.

**Common footgun**: triggering execution unintentionally can dominate runtime.

## 2. Cheap vs Expensive Operations

In pandas, these often feel interchangeable:

```
df.methodA.methodB
df.methodB.methodA
```

With distributed data, order matters much more.

Operations that stay block-local are usually cheap: each block can be processed independently, with no cross-node coordination. These include common ops that are row-independent: `filter`, `select_columns` etc. 

Operations that require global coordination are the expensive ones, especially when they move wide rows across the cluster. Methods like `sort()` and `groupby(...).aggregate(...)` are lazy, but when executed they become expensive global operations.

### Example: wide vs narrow sort

The dataset has a wide schema with a large 'payload_blob' column, but many operations only need the narrow columns. Let's see how much faster it is to sort just the narrow columns instead of the full wide dataset.

In [17]:
narrow_columns = [name for name in events_ds.schema().names if name != 'payload_blob']

# Sorting the wide dataset with the large 'payload_blob' column is much slower than sorting just the narrow columns
wide_events = ray.data.read_parquet(str(input_dir), override_num_blocks=NUM_SHARDS)
start = time.perf_counter()
wide_sorted = wide_events.sort('event_ts').materialize()
wide_elapsed = time.perf_counter() - start
print(f"Sorting wide dataset took {wide_elapsed:.2f} seconds.")
wide_sorted


Parquet dataset sampling: 100%|██████████| 2.00/2.00 [00:00<00:00, 163 file/s]
2026-04-20 09:01:16,942	INFO parquet_datasource.py:1079 -- Estimated parquet encoding ratio is 14.810.
2026-04-20 09:01:16,943	INFO parquet_datasource.py:1139 -- Estimated parquet reader batch size at 65218 rows
2026-04-20 09:01:16,955	INFO logging.py:392 -- Registered dataset logger for dataset dataset_7_0
2026-04-20 09:01:16,958	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_7_0. Full logs are in /tmp/ray/session_2026-04-20_09-01-09_472804_15805/logs/ray-data
2026-04-20 09:01:16,958	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_7_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> AllToAllOperator[Sort]
2026-04-20 09:01:16,977	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_7_0 =======
2026-04-20 09:01:16,978	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-04-20 09:01:16,978	INFO logging_progress.py:227 -- Active & reques

Sorting wide dataset took 0.40 seconds.


shape: (60000, 11)
╭─────────────────────┬───────────┬────────────┬────────┬──────────┬─────────────┬──────────┬────────────┬───────────┬──────────────┬──────────────────────────────────────────╮
│ event_ts            ┆ player_id ┆ session_id ┆ region ┆ platform ┆ event_type  ┆ level_id ┆ latency_ms ┆ spend_usd ┆ is_test_user ┆ payload_blob                             │
│ ---                 ┆ ---       ┆ ---        ┆ ---    ┆ ---      ┆ ---         ┆ ---      ┆ ---        ┆ ---       ┆ ---          ┆ ---                                      │
│ timestamp[ns]       ┆ int32     ┆ int32      ┆ string ┆ string   ┆ string      ┆ int16    ┆ int32      ┆ float     ┆ bool         ┆ string                                   │
╞═════════════════════╪═══════════╪════════════╪════════╪══════════╪═════════════╪══════════╪════════════╪═══════════╪══════════════╪══════════════════════════════════════════╡
│ 2025-01-01 00:00:04 ┆ 50679     ┆ 723835     ┆ LATAM  ┆ console  ┆ chat        ┆ 22       ┆ 24

Sorting the wide dataset with the large 'payload_blob' column is much slower than sorting just the narrow columns, since it has to shuffle much more data across the cluster

In [18]:
narrow_events = ray.data.read_parquet(str(input_dir), override_num_blocks=NUM_SHARDS)
start = time.perf_counter()
narrow_sorted = narrow_events.select_columns(narrow_columns).sort('event_ts').materialize()
narrow_elapsed = time.perf_counter() - start

print(f'wide sort elapsed: {wide_elapsed:.2f} s')
print(f'drop payload -> sort elapsed: {narrow_elapsed:.2f} s')
print(f'wide / narrow ratio: {wide_elapsed / narrow_elapsed:.2f}x')
narrow_sorted

Parquet dataset sampling: 100%|██████████| 2.00/2.00 [00:00<00:00, 244 file/s]
2026-04-20 09:01:17,379	INFO parquet_datasource.py:1079 -- Estimated parquet encoding ratio is 14.810.
2026-04-20 09:01:17,379	INFO parquet_datasource.py:1139 -- Estimated parquet reader batch size at 65218 rows
2026-04-20 09:01:17,395	INFO logging.py:392 -- Registered dataset logger for dataset dataset_12_0
2026-04-20 09:01:17,397	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_12_0. Full logs are in /tmp/ray/session_2026-04-20_09-01-09_472804_15805/logs/ray-data
2026-04-20 09:01:17,397	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_12_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> AllToAllOperator[Sort]
2026-04-20 09:01:17,417	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_12_0 =======
2026-04-20 09:01:17,417	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-04-20 09:01:17,418	INFO logging_progress.py:227 -- Active & re

wide sort elapsed: 0.40 s
drop payload -> sort elapsed: 0.22 s
wide / narrow ratio: 1.83x


shape: (60000, 10)
╭─────────────────────┬───────────┬────────────┬────────┬──────────┬─────────────┬──────────┬────────────┬───────────┬──────────────╮
│ event_ts            ┆ player_id ┆ session_id ┆ region ┆ platform ┆ event_type  ┆ level_id ┆ latency_ms ┆ spend_usd ┆ is_test_user │
│ ---                 ┆ ---       ┆ ---        ┆ ---    ┆ ---      ┆ ---         ┆ ---      ┆ ---        ┆ ---       ┆ ---          │
│ timestamp[ns]       ┆ int32     ┆ int32      ┆ string ┆ string   ┆ string      ┆ int16    ┆ int32      ┆ float     ┆ bool         │
╞═════════════════════╪═══════════╪════════════╪════════╪══════════╪═════════════╪══════════╪════════════╪═══════════╪══════════════╡
│ 2025-01-01 00:00:04 ┆ 50679     ┆ 723835     ┆ LATAM  ┆ console  ┆ chat        ┆ 22       ┆ 242        ┆ 0.0       ┆ False        │
│ 2025-01-01 00:00:16 ┆ 40011     ┆ 706581     ┆ LATAM  ┆ pc       ┆ chat        ┆ 6        ┆ 98         ┆ 0.0       ┆ False        │
│ 2025-01-01 00:00:45 ┆ 69068     ┆ 205518 

In [19]:
print(wide_sorted.stats())
print('\n'+ '=' * 20 + '\n')
print(narrow_sorted.stats())

Operator 1 ReadParquet: 8 tasks executed, 8 blocks produced in 0.09s
* Remote wall time: 15.43ms min, 76.13ms max, 35.3ms mean, 282.38ms total
* Remote cpu time: 6.01ms min, 8.59ms max, 6.87ms mean, 54.98ms total
* UDF time: 0us min, 0us max, 0.0us mean, 0us total
* Peak heap memory usage (MiB): 0.0 min, 0.0 max, 0 mean
* Output num rows per block: 7500 min, 7500 max, 7500 mean, 60000 total
* Output size bytes per block: 15431929 min, 15432845 max, 15432487 mean, 123459900 total
* Output rows per task: 7500 min, 7500 max, 7500 mean, 8 tasks used
* Tasks per node: 8 min, 8 max, 8 mean; 1 nodes used
* Operator throughput:
	* Total input num rows: 0 rows
	* Total output num rows: 60000 rows
	* Ray Data throughput: 683821.3977653628 rows/s
	* Estimated single task throughput: 212478.76002117578 rows/s

Operator 2 Sort: executed in 0.37s

	Suboperator 0 SortMap: 1 tasks executed, 8 blocks produced
	* Remote wall time: 3.67ms min, 5.36ms max, 4.48ms mean, 35.85ms total
	* Remote cpu time: 3.

**Conclusion**: If a later stage does not need a wide column like `payload_blob`, drop it with a block-local op before a global shuffle like `sort()`.

### Example: `groupby` is a communication barrier

Projecting just a few columns is much faster than sorting, since it doesn't require shuffling data across the cluster. This is a common pattern in data processing pipelines, where you want to first filter and project down to a smaller dataset before performing expensive operations like sorting or grouping.

In [20]:
local_events = ray.data.read_parquet(str(input_dir), override_num_blocks=NUM_SHARDS)
start = time.perf_counter()
local_projected = local_events.select_columns(['region', 'latency_ms', 'spend_usd']).materialize()
local_elapsed = time.perf_counter() - start
print(f"Local projection of wide dataset took {local_elapsed:.2f} seconds.")

Parquet dataset sampling: 100%|██████████| 2.00/2.00 [00:00<00:00, 30.5 file/s]
2026-04-20 09:01:17,721	INFO parquet_datasource.py:1079 -- Estimated parquet encoding ratio is 14.810.
2026-04-20 09:01:17,721	INFO parquet_datasource.py:1139 -- Estimated parquet reader batch size at 65218 rows
2026-04-20 09:01:17,748	INFO logging.py:392 -- Registered dataset logger for dataset dataset_16_0
2026-04-20 09:01:17,751	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_16_0. Full logs are in /tmp/ray/session_2026-04-20_09-01-09_472804_15805/logs/ray-data
2026-04-20 09:01:17,751	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_16_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet]
2026-04-20 09:01:17,769	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_16_0 =======
2026-04-20 09:01:17,770	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-04-20 09:01:17,770	INFO logging_progress.py:227 -- Active & requested resources: 0/12 C

Local projection of wide dataset took 0.12 seconds.


In [21]:
grouped_events = ray.data.read_parquet(str(input_dir), override_num_blocks=NUM_SHARDS)
start = time.perf_counter()
region_summary = (
grouped_events
    .select_columns(['region', 'latency_ms', 'spend_usd'])
    .groupby('region')
    .aggregate(
        Count(alias_name='events'),
        Mean('latency_ms', alias_name='mean_latency_ms'),
        Sum('spend_usd', alias_name='total_spend_usd'),
    )
    .materialize()
)
grouped_elapsed = time.perf_counter() - start
print(f"Grouping and aggregation took {grouped_elapsed:.2f} seconds.")

Parquet dataset sampling: 100%|██████████| 2.00/2.00 [00:00<00:00, 135 file/s]
2026-04-20 09:01:17,877	INFO parquet_datasource.py:1079 -- Estimated parquet encoding ratio is 14.810.
2026-04-20 09:01:17,878	INFO parquet_datasource.py:1139 -- Estimated parquet reader batch size at 65218 rows
2026-04-20 09:01:17,907	INFO logging.py:392 -- Registered dataset logger for dataset dataset_22_0
2026-04-20 09:01:17,911	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=8, aggregators=8, dataset (estimate)=0.1GiB): shuffle=14.7MiB, output=14.7MiB, total=29.4MiB, 
2026-04-20 09:01:17,913	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_22_0. Full logs are in /tmp/ray/session_2026-04-20_09-01-09_472804_15805/logs/ray-data
2026-04-20 09:01:17,913	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_22_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> HashAggregateOperator[HashAggregate(key_columns=('r

Grouping and aggregation took 1.80 seconds.


In [22]:
print(f'project only elapsed: {local_elapsed:.2f} s')
print(f'project -> groupby(region) -> aggregate elapsed: {grouped_elapsed:.2f} s')
print(f'groupby / local ratio: {grouped_elapsed / local_elapsed:.2f}x')
display(region_summary.to_pandas().sort_values('region').reset_index(drop=True))

project only elapsed: 0.12 s
project -> groupby(region) -> aggregate elapsed: 1.80 s
groupby / local ratio: 15.44x


,region,events,mean_latency_ms,total_spend_usd
0,APAC,13149,94.100768,26101.919975
1,EU,16145,94.161103,35175.570000
2,LATAM,6135,93.943113,14611.759993
3,MEA,3065,94.057096,7100.019988
4,NA,21506,94.347624,47669.920018


**Conclusion**: `groupby(...).aggregate(...)` is expensive because Ray cannot finish the job block by block in isolation.

Each block can compute partial results for its own rows, but rows with the same key (e.g. region) may live in different blocks. Before Ray can produce the final answer, it has to move data around so that matching keys end up together.

That extra data movement and synchronization is why `groupby` is a global coordination step.


## 3. Summary

This notebook only covered a high-level slice of Ray Data. The full API is broader, and it also works beyond tabular ETL workloads, including text, images, tensors, and batch inference pipelines.

The key ideas to keep are:
- Think in blocks, not individual rows.
- Use block-local ops like `filter`, `select_columns`, and many `map_batches` transforms to shrink data early.
- Methods like `sort()` and `groupby(...).aggregate(...)` are lazy, but when executed they become expensive global operations.
- `materialize()`, `show()`, `sum()`, `to_pandas()`, and often `count()` are the kinds of calls that force execution.
- If a later stage only needs a few columns, project early before the global op.
- Bring data back to pandas only when the result is genuinely small.

For a broader tour of the API, see the [Ray Data docs](https://docs.ray.io/en/latest/data/key-concepts.html) and the [Aggregating Data guide](https://docs.ray.io/en/latest/data/aggregating-data.html).

## 4. Optional Advanced Bonus: Custom Aggregators

You can stop here for the core lesson.

This optional section shows how Ray lets you extend aggregation logic beyond the built-in aggregators.

Custom aggregators follow the same pattern:
1. Look at one block and produce a small partial state.
2. Merge those partial states across blocks.
3. Turn the final state into the answer you want.

Below, the partial state is just `[paid_events, total_events]`, which lets us compute the share of events with non-zero spend in each region.

Custom aggregate function to compute the rate of paid events (events with `spend_usd > 0`) per region.

In [ ]:
class PaidEventRate(AggregateFnV2):
    def __init__(self, on: str = 'spend_usd', ignore_nulls: bool = False, alias_name: str = 'paid_event_rate') -> None:
        super().__init__(alias_name, on=on, ignore_nulls=ignore_nulls, zero_factory=lambda: [0, 0])

    def aggregate_block(self, block: BlockAccessor) -> list[int]:
        pdf = BlockAccessor.for_block(block).to_pandas()  # pdf stands for "Pandas DataFrame"
        spend = pdf[self._target_col_name]
        if self._ignore_nulls:
            spend = spend.dropna()
        paid_events = int(spend.gt(0).sum())
        total_events = int(len(spend))
        return [paid_events, total_events]

    def combine(self, current_accumulator: list[int], new: list[int]) -> list[int]:
        return [current_accumulator[0] + new[0], current_accumulator[1] + new[1]]

    def finalize(self, accumulator: list[int]) -> float:
        paid_events, total_events = accumulator
        return paid_events / total_events if total_events != 0 else np.nan

Calculate the paid event rate by region using the custom aggregate function.

In [24]:
paid_rate_by_region = events_ds.select_columns(['region', 'spend_usd']).groupby('region').aggregate(PaidEventRate(on='spend_usd'))
display(paid_rate_by_region.to_pandas().sort_values('region').reset_index(drop=True))

2026-04-20 09:01:19,723	INFO logging.py:392 -- Registered dataset logger for dataset dataset_26_0
2026-04-20 09:01:19,726	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=8, aggregators=8, dataset (estimate)=0.1GiB): shuffle=14.7MiB, output=14.7MiB, total=29.4MiB, 
2026-04-20 09:01:19,729	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_26_0. Full logs are in /tmp/ray/session_2026-04-20_09-01-09_472804_15805/logs/ray-data
2026-04-20 09:01:19,729	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_26_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> HashAggregateOperator[HashAggregate(key_columns=('region',), num_partitions=8)]
2026-04-20 09:01:19,759	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_26_0 =======
2026-04-20 09:01:19,760	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-04-20 09:01:19,761	INFO logging_progress.py:227 -- Active & requested resources

,region,paid_event_rate
0,APAC,0.067914
1,EU,0.069805
2,LATAM,0.079707
3,MEA,0.072104
4,NA,0.070817
